In [18]:
import pandas as pd

df_train = pd.read_csv("train.csv")
df_test = pd.read_csv("test.csv")

df_train.head()

,CarID,symboling,CarName,fueltype,aspiration,doornumber,carbody,drivewheel,enginelocation,wheelbase,...,enginesize,fuelsystem,boreratio,stroke,compressionratio,horsepower,peakrpm,citympg,highwaympg,price
0,400,0.0,Zenith Core Prime-400,gas,std,four,wagon,fwd,front,99.959654,...,179.0,mpfi,3.417887,3.266419,8.982752,153.0,5193,17.0,22.0,14579.0
1,224,0.0,Voltara Runner Z-224,gas,turbo,four,wagon,4wd,front,96.825986,...,107.0,mpfi,3.624777,2.632518,7.743210,116.0,4804,23.0,23.0,13007.0
2,405,1.0,Voltara Strider Sonic-405,gas,std,two,hatchback,fwd,front,99.088602,...,120.0,2bbl,3.391787,3.397099,8.899534,84.0,4759,26.0,32.0,9933.0
3,103,0.0,Titan Nova Z-103,gas,std,four,wagon,fwd,front,100.400000,...,181.0,mpfi,3.430000,3.270000,9.000000,152.0,5200,17.0,22.0,14399.0
4,109,0.0,Ion Flow Hyper-109,diesel,turbo,four,sedan,rwd,front,107.900000,...,152.0,idi,3.700000,3.520000,21.000000,95.0,4150,28.0,33.0,13200.0


In [19]:
val1 = df_train.enginetype.mode()[0]
val2 = round(df_train[df_train.fueltype == 'gas'].price.mean(), 2)

df_subtasks = pd.DataFrame({
    "subtaskID": [1, 2],
    "datapointID": [1, 1],
    "answer": [val1, val2]
})

df_subtasks

,subtaskID,datapointID,answer
0,1,1,ohc
1,2,1,12728.06


In [20]:
cat_cols = [col for col in df_test.columns if df_test[col].dtype == 'object']
features = [col for col in df_test.columns if col != 'CarID']
target = 'price'

from sklearn.preprocessing import LabelEncoder
for col in cat_cols:
    enc = LabelEncoder()
    enc.fit(pd.concat([df_train[col], df_test[col]]))
    df_train[col] = enc.transform(df_train[col])
    df_test[col] = enc.transform(df_test[col])

df_train = df_train.fillna(0)
df_test = df_test.fillna(0)

df_train.head()

,CarID,symboling,CarName,fueltype,aspiration,doornumber,carbody,drivewheel,enginelocation,wheelbase,...,enginesize,fuelsystem,boreratio,stroke,compressionratio,horsepower,peakrpm,citympg,highwaympg,price
0,400,0.0,645,1,0,0,4,1,0,99.959654,...,179.0,5,3.417887,3.266419,8.982752,153.0,5193,17.0,22.0,14579.0
1,224,0.0,628,1,1,0,4,0,0,96.825986,...,107.0,5,3.624777,2.632518,7.743210,116.0,4804,23.0,23.0,13007.0
2,405,1.0,634,1,0,1,2,1,0,99.088602,...,120.0,1,3.391787,3.397099,8.899534,84.0,4759,26.0,32.0,9933.0
3,103,0.0,542,1,0,0,4,1,0,100.400000,...,181.0,5,3.430000,3.270000,9.000000,152.0,5200,17.0,22.0,14399.0
4,109,0.0,245,0,1,0,3,2,0,107.900000,...,152.0,3,3.700000,3.520000,21.000000,95.0,4150,28.0,33.0,13200.0


In [21]:
X = df_train[features]
y = df_train[target]
X_final = df_test[features]

X.head()

,symboling,CarName,fueltype,aspiration,doornumber,carbody,drivewheel,enginelocation,wheelbase,carlength,...,cylindernumber,enginesize,fuelsystem,boreratio,stroke,compressionratio,horsepower,peakrpm,citympg,highwaympg
0,0.0,645,1,0,0,4,1,0,99.959654,183.678746,...,3,179.0,5,3.417887,3.266419,8.982752,153.0,5193,17.0,22.0
1,0.0,628,1,1,0,4,0,0,96.825986,173.140507,...,2,107.0,5,3.624777,2.632518,7.743210,116.0,4804,23.0,23.0
2,1.0,634,1,0,1,2,1,0,99.088602,177.409929,...,2,120.0,1,3.391787,3.397099,8.899534,84.0,4759,26.0,32.0
3,0.0,542,1,0,0,4,1,0,100.400000,184.600000,...,3,181.0,5,3.430000,3.270000,9.000000,152.0,5200,17.0,22.0
4,0.0,245,0,1,0,3,2,0,107.900000,186.700000,...,2,152.0,3,3.700000,3.520000,21.000000,95.0,4150,28.0,33.0


In [22]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score

model = RandomForestRegressor()

cross_val_score(model, X, y, cv=3, scoring='r2').mean()

0.9714752958377768

In [23]:
model.fit(X, y)
y_preds = model.predict(X_final)

In [24]:
df_preds = pd.DataFrame({
    "subtaskID": 3,
    "datapointID": df_test.CarID,
    "answer": y_preds
})

df_preds.head()

,subtaskID,datapointID,answer
0,3,436,19162.32
1,3,441,15333.69
2,3,553,6348.44
3,3,351,7479.99
4,3,702,7012.99


In [25]:
submission = pd.concat([df_subtasks, df_preds])
submission = submission.reset_index(drop=True)
submission

,subtaskID,datapointID,answer
0,1,1,ohc
1,2,1,12728.06
2,3,436,19162.32
3,3,441,15333.69
4,3,553,6348.44
...,...,...,...
279,3,172,12936.39
280,3,564,7068.95
281,3,335,34373.85
282,3,108,15278.86


In [26]:
submission.to_csv("output.csv", index=False)